In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по овцы и козы v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Овцы и козы
2171,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-03-01,4921.76
1871,ПАВЛОДАРСКАЯ ОБЛАСТЬ,2016-12-01,1386.48
927,ГШЫМКЕНТ,2021-11-01,52.30
108,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-01-01,544.33
1567,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2022-04-01,484.98
1437,КОСТАНАЙСКАЯ ОБЛАСТЬ,2022-01-01,446.01
837,ГАСТАНА,2021-07-01,1.40
139,АКТЮБИНСКАЯ ОБЛАСТЬ,2016-01-01,1470.52
1572,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2022-09-01,685.81
1387,КОСТАНАЙСКАЯ ОБЛАСТЬ,2017-11-01,758.87


In [3]:
# === загружаем данные ===
best_methods = pd.read_excel("results/овцы и козы - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,10.06,76.07,1.012000e+01,77.20,10.47,80.90,HW,MAPE,1.006000e+01,76.07
1,АКТЮБИНСКАЯ ОБЛАСТЬ,4.04,97.03,4.410000e+00,97.54,9.52,182.57,HW,MAPE,4.040000e+00,97.03
2,АТЫРАУСКАЯ ОБЛАСТЬ,7.38,76.98,9.420000e+00,95.38,8.01,102.92,HW,MAPE,7.380000e+00,76.98
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,6.68,110.22,6.980000e+00,93.14,27.39,328.34,HW,MAPE,6.680000e+00,93.14
4,ГАЛМАТЫ,NaN,0.53,7.135685e+21,0.58,NaN,0.58,HW,MAE,7.135685e+21,0.53
5,ГШЫМКЕНТ,12.97,10.31,2.615000e+01,24.08,14.11,16.04,HW,MAPE,1.297000e+01,10.31
6,ЖАМБЫЛСКАЯ ОБЛАСТЬ,3.40,104.40,3.760000e+00,107.30,7.66,235.79,HW,MAPE,3.400000e+00,104.40
7,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,1.59,18.57,3.390000e+00,52.58,7.25,96.05,HW,MAPE,1.590000e+00,18.57
8,ОБЛАСТЬ АБАЙ,8.87,161.85,9.910000e+00,179.71,22.80,508.75,HW,MAPE,8.870000e+00,161.85
9,ОБЛАСТЬ ЖЕТІСУ,5.67,115.60,6.130000e+00,118.59,17.17,607.89,HW,MAPE,5.670000e+00,115.60


In [5]:
actual_aug = pd.read_excel("Овцы и козы 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Овцы и козы"] = (actual_aug["Овцы и козы"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Овцы и козы обработанные август 2025.xlsx", index=False)
actual_aug



,Регион,Период,Овцы и козы
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-08-01,511.19
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2025-08-01,1450.97
2,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-08-01,2306.74
3,АТЫРАУСКАЯ ОБЛАСТЬ,2025-08-01,1023.41
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-08-01,1816.98
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-08-01,2181.60
6,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2025-08-01,878.56
7,КОСТАНАЙСКАЯ ОБЛАСТЬ,2025-08-01,143.59
8,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2025-08-01,582.02
9,МАНГИСТАУСКАЯ ОБЛАСТЬ,2025-08-01,178.80


In [6]:
# === настройки ===
TARGET = "Овцы и козы"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [7]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [8]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [9]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [10]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [11]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [12]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Овцы и козы - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

,Регион,Лучший метод,Прогноз (2025-08),Факт (2025-08),"Отклонение, %"
0,АКМОЛИНСКАЯ ОБЛАСТЬ,HW,526.93,511.19,3.08
1,АКТЮБИНСКАЯ ОБЛАСТЬ,HW,1600.01,1450.97,10.27
2,АЛМАТИНСКАЯ ОБЛАСТЬ,SARIMA,2541.15,2306.74,10.16
3,АТЫРАУСКАЯ ОБЛАСТЬ,HW,965.75,1023.41,-5.63
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,HW,725.14,886.84,-18.23
5,ГАЛМАТЫ,HW,1.06,NaN,NaN
6,ГАСТАНА,SARIMA,1.99,NaN,NaN
7,ГШЫМКЕНТ,HW,58.67,NaN,NaN
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,HW,2280.95,2181.60,4.55
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,HW,1824.03,1816.98,0.39
